# Ch?y Lab 2?10 v? l?u k?t qu? l?n GitHub

1. Ch?n **Runtime ? Change runtime type ? T4 GPU**.
2. T?o fine-grained token GitHub, ch? ch?n repo `hungnt1402/advancedhpc2026`, c?p **Contents: Read and write**.
3. Trong Colab m? **Secrets** (bi?u t??ng ch?a kh?a), th?m `GITHUB_TOKEN`, b?t **Notebook access**. Kh?ng d?n token v?o code ho?c output.
4. N?u d?ng ?nh ri?ng, t?i `image.jpg` v? (Lab 6) `image2.jpg` v?o `/content` tr??c khi ch?y.
5. Ch?n **Runtime ? Run all**. M?i lab ho?n t?t ???c commit v? push t? ??ng v?o `results/<th?i ?i?m UTC>/LabN/` tr?n nh?nh `main`.

K?t qu? g?m notebook ?? ch?y, log, th?ng tin GPU, ?nh ??u v?o/??u ra, CSV, m?ng NumPy n?u c?, LaTeX v? PDF. Lab 2 l?u th?ng tin thi?t b?. Lab l?i ???c ghi trong `summary.json`; notebook l?i gi? trong th? m?c ch?n ?o?n c?a phi?n Colab. C?c lab kh?c v?n ti?p t?c. N?u push l?i, ch??ng tr?nh d?ng v? gi? k?t qu? trong Files ?? t?i xu?ng.

Colab c? th? ng?t phi?n ho?c h?t h?n m?c GPU; m? l?i v? ??t `LABS` th?nh c?c lab c?n thi?u ?? ch?y ti?p. M?i l?n ch?y t?o th? m?c m?i. B?n notebook trong `LabN/` gi? nguy?n.


In [ ]:
LABS = list(range(2, 11))  # V? d? ch?y l?i lab l?i: [5, 10]
COMPILE_PDF = True
REPO = "/content/advancedhpc2026"


In [ ]:
%pip -q install "numba-cuda[cu12]" nbclient nbformat ipykernel pandas pillow matplotlib
import subprocess
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
if COMPILE_PDF:
    with open("/content/latex-install.log", "w") as log:
        subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=log, stderr=log)
        subprocess.run([
            "apt-get", "install", "-y", "-qq", "texlive-latex-base",
            "texlive-latex-recommended", "texlive-fonts-recommended", "texlive-lang-other"
        ], check=True, stdout=log, stderr=log)
    print("LaTeX ready")


In [ ]:
from google.colab import userdata

# Ch? ??c secret trong b? ?i?u khi?n; kh?ng chuy?n token v?o kernel c?a c?c lab.
_token = userdata.get("GITHUB_TOKEN")
if not _token:
    raise RuntimeError("Th?m GITHUB_TOKEN v?o Secrets v? b?t Notebook access.")
repo_path = Path(REPO)
if not repo_path.exists():
    subprocess.run([
        "git", "clone", "--branch", "main",
        "https://github.com/hungnt1402/advancedhpc2026.git", REPO
    ], check=True)
else:
    if subprocess.check_output(["git", "-C", REPO, "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Checkout c?n thay ??i. T?i k?t qu? ch?a push trong Files tr??c khi ??ng b?.")
    subprocess.run(["git", "-C", REPO, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "config", "user.name", "Colab Lab Runner"], check=True)
subprocess.run(["git", "-C", REPO, "config", "user.email", "colab-runner@users.noreply.github.com"], check=True)


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("colab_run_all", repo_path / "scripts/colab_run_all.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
try:
    summary = runner.run_all(REPO, _token, labs=LABS, compile_pdf=COMPILE_PDF)
finally:
    del _token
